# Notebook 02 — Feature Engineering

Walk through the 25-feature lexical extractor:
- Demonstrate feature extraction on example URLs
- Compare feature values for phishing vs legitimate URLs
- Feature importance preview (mutual information)
- Save processed datasets as Parquet for faster loading

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import extract_features, extract_features_batch, FEATURE_NAMES
from src.data_loader import load_d1, load_d2, generate_synthetic_dataset

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})
print('Ready.')

## 1. Feature Extraction Demo

In [ ]:
DEMO_URLS = [
    # Phishing
    'http://paypal-login.ru/verify/account?id=abc123&token=xyz',
    'http://192.168.1.1/bank/update/credentials.php',
    'http://secure-amazon.tk/signin/confirm/billing',
    'http://bit.ly/3xPhishURL',
    # Legitimate
    'https://www.google.com/search?q=weather+london',
    'https://en.wikipedia.org/wiki/Phishing',
    'https://github.com/scikit-learn/scikit-learn',
    'https://www.amazon.com/dp/B09G9FPHY6',
]

df_demo = pd.DataFrame([extract_features(url) for url in DEMO_URLS])
df_demo.insert(0, 'url', DEMO_URLS)
df_demo.insert(1, 'label', ['Phishing']*4 + ['Legitimate']*4)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
display(df_demo.set_index(['url', 'label']).T)

## 2. Feature Descriptions

In [ ]:
descriptions = {
    'url_length':         'Total character count of the full URL',
    'domain_length':      'Character count of the netloc (host+port)',
    'path_length':        'Character count of the URL path',
    'query_length':       'Character count of the query string',
    'n_dots':             'Number of "." characters in the URL',
    'n_hyphens':          'Number of "-" characters',
    'n_underscores':      'Number of "_" characters',
    'n_slashes':          'Number of "/" characters',
    'n_at':               'Number of "@" characters (credential injection)',
    'n_digits':           'Total digit count in the URL',
    'n_special_chars':    'Non-alphanumeric, non-standard characters',
    'has_ip':             '1 if IP address used instead of domain name',
    'has_port':           '1 if a non-default port is specified',
    'is_https':           '1 if scheme is HTTPS',
    'subdomain_level':    'Number of subdomain labels',
    'path_depth':         'Number of path segments',
    'url_entropy':        'Shannon entropy of the full URL string',
    'domain_entropy':     'Shannon entropy of the domain portion',
    'has_suspicious_word':'1 if any phishing keyword found in URL',
    'n_suspicious_words': 'Count of phishing keywords present',
    'uses_shortener':     '1 if URL shortener domain detected',
    'has_redirect':       '1 if "//" appears after the protocol section',
    'tld_is_common':      '1 if TLD is .com/.org/.net/.edu/.gov etc.',
    'brand_in_subdomain': '1 if a known brand name appears in subdomain',
    'brand_in_path':      '1 if a known brand name appears in the path',
}

desc_df = pd.DataFrame.from_dict(
    descriptions, orient='index', columns=['Description']
).rename_axis('Feature')
display(desc_df)

## 3. Mutual Information — Feature Relevance Ranking

In [ ]:
import os
from sklearn.feature_selection import mutual_info_classif

D1_PATH = '../data/raw/dataset_phishing_uci.csv'
if os.path.exists(D1_PATH):
    X1, y1 = load_d1(D1_PATH, use_lexical=True)
else:
    X1, y1 = generate_synthetic_dataset(n_phishing=600, n_legit=600)

mi_scores = mutual_info_classif(X1, y1, random_state=42)
mi_df = pd.DataFrame({'feature': FEATURE_NAMES, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=True)

fig, ax = plt.subplots(figsize=(8, 9))
colors = ['#e63946' if v > mi_df['mi_score'].median() else '#4361ee'
          for v in mi_df['mi_score']]
ax.barh(mi_df['feature'], mi_df['mi_score'], color=colors, edgecolor='white')
ax.axvline(mi_df['mi_score'].median(), color='black', linestyle='--',
            linewidth=1.2, label='Median MI')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Relevance (Mutual Information)\nD1 — UCI Dataset', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/feature_mutual_info.png', bbox_inches='tight')
plt.show()
print('Saved: feature_mutual_info.png')

## 4. Pairplot of Top-6 Features

In [ ]:
top6 = mi_df.tail(6)['feature'].tolist()

df_plot = X1[top6].copy()
df_plot['label'] = y1.values
df_plot['Class'] = df_plot['label'].map({0: 'Legitimate', 1: 'Phishing'})

g = sns.pairplot(
    df_plot, vars=top6, hue='Class',
    palette={'Legitimate': '#2ec4b6', 'Phishing': '#e63946'},
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kind='kde',
)
g.fig.suptitle('Pairplot — Top 6 Informative Features', y=1.01, fontweight='bold')
g.fig.savefig('../results/figures/feature_pairplot_top6.png', bbox_inches='tight')
plt.show()
print('Saved: feature_pairplot_top6.png')

## 5. Save Processed Datasets

In [ ]:
import os

D2_PATH = '../data/raw/phiusiil_phishing_url.csv'

os.makedirs('../data/processed', exist_ok=True)

# D1
df1 = X1.copy()
df1['label'] = y1.values
df1.to_parquet('../data/processed/d1_features.parquet', index=False)
print(f'D1 saved → data/processed/d1_features.parquet ({len(df1):,} rows)')

# D2
if os.path.exists(D2_PATH):
    X2, y2 = load_d2(D2_PATH, sample=50000)
    df2 = X2.copy()
    df2['label'] = y2.values
    df2.to_parquet('../data/processed/d2_features.parquet', index=False)
    print(f'D2 saved → data/processed/d2_features.parquet ({len(df2):,} rows)')
else:
    print('D2 CSV not found — skipping D2 save.')